<a href="https://colab.research.google.com/github/elkins/synth-saxs/blob/main/examples/interactive_tutorials/hydration_shell_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Interactive Hydration Shell Analysis

In Small-Angle X-ray Scattering (SAXS), the protein is not a "dry" object in a vacuum. It is surrounded by solvent (water). One of the most subtle but important effects in SAXS modeling is the **hydration shell**—a layer of water molecules immediately surrounding the protein that is slightly denser (~2-5%) than bulk water.

This tutorial demonstrates how this "excess" density affects the scattering profile and the perceived size of the protein.

In [ ]:
import sys

# Install dependencies if running in Colab or a new environment
if "google.colab" in sys.modules:
    !pip install -q synth-saxs biotite matplotlib ipywidgets

import biotite.database.rcsb as rcsb
import biotite.structure.io as strucio
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, interact

from synth_saxs import calculate_radius_of_gyration, calculate_saxs_profile

print("Setup complete.")

## 1. Load a Reference Structure
We will use **Ubiquitin (1UBQ)**, a small and well-studied protein.

In [ ]:
# Download and load 1UBQ
pdb_file = rcsb.fetch("1UBQ", "pdb", ".")
structure = strucio.load_structure(pdb_file)
if hasattr(structure, "stack_depth") and structure.stack_depth() > 1:
    structure = structure[0]

# Pre-calculate the Radius of Gyration (dry)
rg_dry = calculate_radius_of_gyration(structure)
print(f"Structure loaded. Dry Rg: {rg_dry:.2f} A")

## 2. Interactive Visualization

The hydration shell is modeled by adding an excess density $\Delta\rho_{shell}$ to the solvent term. 

The effective scattering factor $f_{eff}(q)$ for an atom is roughly:
$$ f_{eff}(q) = f_{vac}(q) - (\rho_{sol} - \Delta\rho_{shell}) \cdot V \cdot \exp(-q^2 R^2 / 10) $$

As $\Delta\rho_{shell}$ increases, the "contrast" between the protein and its immediate surroundings changes, making the protein appear larger or "brighter" to X-rays.

In [ ]:
def update_plot(shell_density):
    # 1. Calculate profiles
    q, i_bulk = calculate_saxs_profile(structure, hydration_shell_density=0.0, n_points=100)
    _, i_shell = calculate_saxs_profile(
        structure, hydration_shell_density=shell_density, n_points=100
    )

    # 2. Plotting
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Log-linear plot
    ax1.semilogy(q, i_bulk, "k--", label="Bulk Solvent only", alpha=0.5)
    ax1.semilogy(q, i_shell, "r-", linewidth=2, label=f"Shell Density: {shell_density:.3f} e/A^3")
    ax1.set_xlabel("q (A^-1)", fontsize=12)
    ax1.set_ylabel("log I(q)", fontsize=12)
    ax1.set_title("Effect on Scattering Intensity", fontsize=14)
    ax1.legend()

    # Difference plot (normalized to bulk at q=0)
    diff = (i_shell - i_bulk) / i_bulk[0] * 100
    ax2.plot(q, diff, "g-")
    ax2.set_xlabel("q (A^-1)", fontsize=12)
    ax2.set_ylabel("% Difference (relative to I(0))", fontsize=12)
    ax2.set_title("Relative Intensity Increase", fontsize=14)
    ax2.grid(True, linestyle="--", alpha=0.7)

    plt.tight_layout()
    plt.show()


# Create the interactive slider
interact(
    update_plot,
    shell_density=FloatSlider(
        value=0.03,
        min=0.0,
        max=0.06,
        step=0.005,
        description="Shell Dens:",
        continuous_update=False,
    ),
);

### What are you seeing?

1. **Increased Contrast:** Adding a hydration shell increases the scattering intensity at $q=0$. This is because the "effective volume" of the protein is increasing as it pulls in a layer of denser water.
2. **The "Boom" at low q:** Notice that the relative difference is most pronounced at low $q$ values. This is where the global shape (the "envelope") of the protein dominates the scattering.
3. **Experimental Reality:** In real experiments, the hydration shell density is typically between **0.02 and 0.05 e/Å³**. If you ignore this effect, your simulated curves will often systematically underestimate the experimental intensity at low angles.